<a href="https://colab.research.google.com/github/eshikanahata/DC-Mini-Project/blob/Nikhil/task5_with_agentic_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Section 0: Setup

Install and import everything you need. This builds on Task 3's dependencies.

In [ ]:
# Install all required libraries
# Run this cell once. It may take a few minutes.

!pip install -q -U \
  torch \
  transformers \
  sentence-transformers \
  accelerate \
  langchain \
  langchain-community \
  chromadb \
  pysqlite3-binary \
  openai \
  pandas \
  scikit-learn \
  tqdm

print("Installation complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
!pip uninstall -y torchcodec

Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128


In [ ]:
# Fix Colab's SQLite version issue (must run before importing chromadb)
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

# Core imports
import os
import json
import pandas as pd
from tqdm import tqdm
import torch

# LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter   # ← fixed
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Check GPU
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Embeddings will be slow. Switch to T4 GPU runtime.")

In [ ]:
!pip install -q transformers accelerate

In [ ]:
# ─────────────────────────────────────────────
# LLM SETUP — Uncomment ONE option below
# ─────────────────────────────────────────────

# OPTION A: OpenAI (requires API key)
# from openai import OpenAI
# os.environ["OPENAI_API_KEY"] = "sk-..."
# llm_client = OpenAI()

# OPTION B: Anthropic Claude (requires API key)
# !pip install -q anthropic
# import anthropic
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# llm_client = anthropic.Anthropic()

# OPTION C: Flan-T5 — lightweight, CPU-friendly, weak reasoning
# from transformers import pipeline
# llm_pipeline = pipeline(
#     "text2text-generation",          # ← must be text2text, NOT text-generation
#     model="google/flan-t5-large",
#     device=0 if torch.cuda.is_available() else -1,
# )

# OPTION D: Qwen3 — strongest free local option, needs T4 GPU
# Qwen3-1.7B fits on Colab T4; use 0.6B if you hit OOM
from transformers import pipeline
llm_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen3-1.7B",          # swap to Qwen/Qwen2.5-1.5B-Instruct if Qwen3 is slow
    device_map="auto",
    torch_dtype=torch.float16,        # halves VRAM usage on GPU
    max_new_tokens=512,
)
print("LLM loaded.")

# NOTE: If you switch options, update generate_verdict() in Section 2 to match.

In [ ]:
# Step 2: Reinstall torchvision to match — replace cu130 with whatever Step 1 showed
!pip install -q --upgrade torchvision --index-url https://download.pytorch.org/whl/cu130

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 61.4 MB/s eta 0:00:00


In [ ]:
# Step 3: Restart the runtime after install
import os
os.kill(os.getpid(), 9)  # Force restart — you MUST rerun all cells after this

---
## Section 1: Rebuild Your Pipeline from Task 3

Before building the verdict system, we need to reload the story and rebuild the vector database.

In [ ]:
# ─────────────────────────────────────────────
# 1A: Load both story texts
# ─────────────────────────────────────────────

from google.colab import files

print("Upload both book .txt files when prompted...")
uploaded = files.upload()  # Upload both at once — Colab allows multi-select

BOOK_PATHS = {
    "The Count of Monte Cristo":  "The Count of Monte Cristo.txt",
    "In Search of the Castaways": "In Search of the Castaways.txt",
}

raw_stories = {}
for book_name, path in BOOK_PATHS.items():
    with open(path, 'r', encoding='utf-8') as f:
        raw_stories[book_name] = f.read()
    print(f"Loaded '{book_name}': {len(raw_stories[book_name]):,} characters.")

In [ ]:
# ─────────────────────────────────────────────
# 1B: Chunk the story using Recursive Splitter
# ─────────────────────────────────────────────

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# 1B: Chunk both stories
all_chunks = {}
for book_name, text in raw_stories.items():
    all_chunks[book_name] = splitter.split_text(text)
    print(f"'{book_name}': {len(all_chunks[book_name])} chunks.")

#Explore other chunking techniques...what about semantic chunking?

'The Count of Monte Cristo': 9646 chunks.
'In Search of the Castaways': 3068 chunks.


In [ ]:
# ─────────────────────────────────────────────
# 1C: Embed chunks and store in ChromaDB (one DB per book)
# ─────────────────────────────────────────────

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

VECTOR_DB_PATHS = {
    "The Count of Monte Cristo":  "./vectordb_monte_cristo",
    "In Search of the Castaways": "./vectordb_castaways",
}

vector_dbs = {}
for book_name, db_path in VECTOR_DB_PATHS.items():
    chunks = all_chunks[book_name]  # from 1B
    vector_dbs[book_name] = Chroma.from_texts(
        texts=chunks,
        embedding=embedding_model,
        persist_directory=db_path,
    )
    print(f"'{book_name}' DB: {vector_dbs[book_name]._collection.count()} chunks.")

/tmp/ipykernel_2046/3767989493.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

'The Count of Monte Cristo' DB: 9646 chunks.
'In Search of the Castaways' DB: 3068 chunks.


In [ ]:
# ─────────────────────────────────────────────
# 1D: Retrieval function — routes to the correct book's vector DB
# ─────────────────────────────────────────────

def retrieve_context(query: str, book_name: str, char: str = "",
                     caption: str = "", k: int = 5) -> list[str]:
    parts = [p for p in [char, caption, query] if p and str(p) != "nan"]
    search_query = " ".join(parts)

    db = vector_dbs[book_name]
    results = db.similarity_search(search_query, k=k * 3)

    seen = set()
    unique_chunks = []
    for doc in results:
        content = " ".join(doc.page_content.split())
        if content not in seen:
            seen.add(content)
            unique_chunks.append(doc.page_content)
        if len(unique_chunks) == k:
            break
    return unique_chunks


# Quick sanity check — one query per book
for book, query in [
    ("The Count of Monte Cristo",  "Dantès captain ship Morrel"),
    ("In Search of the Castaways", "Thalcave guide pampas"),
]:
    results = retrieve_context(query, book_name=book)
    print(f"\n[{book}] Query: '{query}'")
    for i, chunk in enumerate(results):
        print(f"  --- Chunk {i+1} ---\n  {chunk[:200]}...")


[The Count of Monte Cristo] Query: 'Dantès captain ship Morrel'
  --- Chunk 1 ---
  “Now, if you will come on board, M. Morrel,” said Dantès, observing the
owner’s impatience, “here is your supercargo, M. Danglars, coming out
of his cabin, who will furnish you with every particular. ...
  --- Chunk 2 ---
  “I am entirely at your service, M. Morrel,” answered Danglars. “You
know that I am as capable of managing a ship as the most experienced
captain in the service; and it will be so far advantageous to y...
  --- Chunk 3 ---
  “Yes,” continued Caderousse, “so it is; after five-and-twenty years of
labor, after having acquired a most honorable name in the trade of
Marseilles, M. Morrel is utterly ruined; he has lost five ship...
  --- Chunk 4 ---
  In fact, a moment later M. Morrel appeared and was saluted with an
enthusiastic burst of applause from the crew of the _Pharaon_, who
hailed the visit of the shipowner as a sure indication that the ma...
  --- Chunk 5 ---
  failed him, and he 

---
## Section 2: The MVP - Claim Verification Pipeline
**Week 4 Goal:** Build the core function that takes a claim and outputs `1` (Consistent) or `0` (Inconsistent).

In [ ]:
# ─────────────────────────────────────────────
# 2A: The Standard Prompt (Baseline)
# ─────────────────────────────────────────────
# This is the simplest possible prompt. We will compare it against
# a Chain-of-Thought prompt in Section 4.

def build_standard_prompt(claim: str, context_chunks: list[str]) -> str:
    """
    Builds a simple yes/no prompt.

    Args:
        claim: The backstory claim to verify.
        context_chunks: Retrieved story passages.

    Returns:
        A formatted prompt string.
    """
    context_str = "\n\n".join([f"[Passage {i+1}]: {chunk}" for i, chunk in enumerate(context_chunks)])

    prompt = f"""You are a story fact-checker. Read the passages below and decide if the claim is consistent with them.

STORY PASSAGES:
{context_str}

CLAIM: "{claim}"

Is this claim consistent with the story passages? Answer with only '1' for Consistent or '0' for Inconsistent.
Answer:"""

    return prompt


# Preview what the prompt looks like
sample_claim = "Fernand involved in the betrayal of Dantès?"
sample_context = retrieve_context(sample_claim, book_name="The Count of Monte Cristo")
print(build_standard_prompt(sample_claim, sample_context))

You are a story fact-checker. Read the passages below and decide if the claim is consistent with them.

STORY PASSAGES:
[Passage 1]: vengeance. Fernand’s mind was made up; he would shoot Dantès, and then
kill himself. But Fernand was mistaken; a man of his disposition never
kills himself, for he constantly hopes.

[Passage 2]: “I know not why you meddle,” said Fernand, seizing his arm; “but this I
know, you have some motive of personal hatred against Dantès, for he
who himself hates is never mistaken in the sentiments of others.”

[Passage 3]: “Ah, _ma foi_, under any circumstances!” said Caderousse, who drank as
he spoke, and on whom the fumes of the wine began to take
effect,—“under any circumstances Fernand is not the only person put out
by the fortunate arrival of Dantès; is he, Danglars?”

“No, you are right—and I should say that would bring him ill-luck.”

[Passage 4]: 0056m



“Yes; but one gets out of prison,” said Caderousse, who, with what
sense was left him, listened eagerly

In [ ]:
# ─────────────────────────────────────────────
# 2B: LLM Call Function
# ─────────────────────────────────────────────
# This wraps your chosen LLM (from Section 0) in a single callable.
# If you chose Option A or B in Section 0, update the logic inside this function.

def call_llm(prompt: str, max_new_tokens: int = 50) -> str:
    """
    Sends a prompt to the LLM and returns the raw text response.

    TODO: If using Option A (OpenAI), replace the body with:
        response = llm_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50
        )
        return response.choices[0].message.content.strip()

    TODO: If using Option B (Anthropic), replace the body with:
        message = llm_client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=50,
            messages=[{"role": "user", "content": prompt}]
        )
        return message.content[0].text.strip()
    """
    # Default: HuggingFace local model (Option C)
    result = llm_pipeline(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    return result[0]['generated_text'].replace(prompt, '').strip()


# Test the LLM call
test_response = call_llm("Answer with only the number 1 or 0. Is 2+2=4? Answer:")
print(f"Test LLM response: '{test_response}'")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test LLM response: '1

Is 2+2=4? Answer: 1

Is 2+2=4? Answer: 1

Is 2+2=4? Answer: 1

Is 2+2=4?'


In [ ]:
# ─────────────────────────────────────────────
# 2C: Parse the verdict from the LLM's response
# ─────────────────────────────────────────────
# LLMs don't always return a clean "1" or "0".
# This function extracts a clean binary verdict from messy text output.

def parse_verdict(llm_response: str) -> int:
    """
    Parses the LLM response into 1 (Consistent) or 0 (Inconsistent).
    Returns -1 if the verdict could not be determined.

    Args:
        llm_response: Raw text from the LLM.

    Returns:
        1, 0, or -1 (undecided).
    """
    # TODO: This is a naive parser. Can you make it more robust?
    # Hint: Look for keywords like "consistent", "inconsistent", "yes", "no"
    # in addition to the digits 1 and 0.

    response_lower = llm_response.lower().strip()

    if '1' in response_lower[:10] or 'consistent' in response_lower and 'inconsistent' not in response_lower:
        return 1
    elif '0' in response_lower[:10] or 'inconsistent' in response_lower:
        return 0
    else:
        return -1  # Could not parse


# Test parser
print(parse_verdict("1"))            # Expected: 1
print(parse_verdict("0"))            # Expected: 0
print(parse_verdict("Consistent"))   # Expected: 1
print(parse_verdict("Inconsistent")) # Expected: 0
print(parse_verdict("I'm not sure")) # Expected: -1

1
0
1
0
-1


In [ ]:
# ─────────────────────────────────────────────
# 2D: The Full MVP Pipeline — putting it all together
# ─────────────────────────────────────────────

def verify_claim(claim: str, book_name: str, k: int = 3, prompt_type: str = "standard", verbose: bool = False) -> dict:
    context_chunks = retrieve_context(claim, book_name=book_name, k=k)
    # everything else identical to original

    # Step 2: Build the prompt
    if prompt_type == "cot":
        # TODO: Will be implemented in Section 4
        prompt = build_cot_prompt(claim, context_chunks)
    else:
        prompt = build_standard_prompt(claim, context_chunks)

    if verbose:
        print("=" * 60)
        print(f"CLAIM: {claim}")
        print("-" * 60)
        print("RETRIEVED CONTEXT:")
        for i, c in enumerate(context_chunks):
            print(f"  [{i+1}] {c[:120]}...")
        print("-" * 60)

    # Step 3: Call the LLM
    raw_response = call_llm(prompt)

    if verbose:
        print(f"RAW LLM RESPONSE: {raw_response}")

    # Step 4: Parse the verdict
    verdict = parse_verdict(raw_response)

    if verbose:
        verdict_label = {1: "CONSISTENT ", 0: "INCONSISTENT ", -1: "UNDECIDED "}
        print(f"VERDICT: {verdict_label.get(verdict, '?')}")
        print("=" * 60)

    return {
        "claim": claim,
        "verdict": verdict,
        "raw_response": raw_response,
        "context": context_chunks
    }


# ─── Test the MVP ────────────────────────────
# Run both a consistent and inconsistent claim to verify the pipeline works.

result_1 = verify_claim(
    claim="Fernand involved in the betrayal of Dantès?",
    book_name = "The Count of Monte Cristo",
    verbose=True
)

result_2 = verify_claim(
    claim="Dantès imprisoned for 30 years?",
    book_name = "The Count of Monte Cristo",
    verbose=True
)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CLAIM: Fernand involved in the betrayal of Dantès?
------------------------------------------------------------
RETRIEVED CONTEXT:
  [1] vengeance. Fernand’s mind was made up; he would shoot Dantès, and then
kill himself. But Fernand was mistaken; a man of ...
  [2] “I know not why you meddle,” said Fernand, seizing his arm; “but this I
know, you have some motive of personal hatred ag...
  [3] “Ah, _ma foi_, under any circumstances!” said Caderousse, who drank as
he spoke, and on whom the fumes of the wine began...
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAW LLM RESPONSE: 1
The claim that "Fernand involved in the betrayal of Dantès" is consistent with the story passages. 

Passage 1: Fernand's mind is made up to shoot Dantès and then kill himself, but
VERDICT: CONSISTENT 
CLAIM: Dantès imprisoned for 30 years?
------------------------------------------------------------
RETRIEVED CONTEXT:
  [1] “Yes, but they will make you then sign your declaration, and confront
you with him you have denounced; I will supply you...
  [2] Dantès shuddered; this man had been four years longer than himself in
prison.

“Do not dig any more,” said the voice; “o...
  [3] 0056m



“Yes; but one gets out of prison,” said Caderousse, who, with what
sense was left him, listened eagerly to the ...
------------------------------------------------------------
RAW LLM RESPONSE: 0
The claim that "Dantès was imprisoned for 30 years" is inconsistent with the story passages. The passages indicate that Dantès was imprisoned for four years, as stated in Passage 2, where 

---
## Section 3: Evaluation - Measuring Your Baseline Accuracy
**Week 5 Goal:** You can't improve what you don't measure. Run your MVP against a test dataset and compute accuracy.

> **Key Insight:** The test cases you create here will tell you *where your system fails* - which is more valuable than knowing where it succeeds.

In [ ]:
# ─────────────────────────────────────────────
# 3A: Build your evaluation dataset
# ─────────────────────────────────────────────
# Each entry has:
#   - claim: the statement to verify
#   - ground_truth: 1 (consistent with the story) or 0 (inconsistent)
#   - note: why this case is interesting or tricky

# TODO: Add at least 10 test cases based on YOUR story.
# Include a mix of:
#   - Easy consistent claims (direct facts from the story)
#   - Easy inconsistent claims (clear contradictions)
#   - Hard cases (subtle contradictions, things not mentioned, paraphrased facts)

#Note : The dataset which I shared contains train and test csv files both, this is just a sample so you guys understand the notebook first, then when you check
#out the dataset it does not feel something new or not familiar.
train_df = pd.read_csv("train.csv")
train_df["ground_truth"] = train_df["label"].map({"consistent": 1, "contradict": 0})

print(f"Evaluation dataset ready: {len(train_df)} test cases.")
print(f"  Consistent claims:   {(train_df['ground_truth'] == 1).sum()}")
print(f"  Inconsistent claims: {(train_df['ground_truth'] == 0).sum()}")

Evaluation dataset ready: 80 test cases.
  Consistent claims:   51
  Inconsistent claims: 29


In [ ]:
# ─────────────────────────────────────────────
# 3B: Run the evaluation loop
# ─────────────────────────────────────────────

def run_evaluation(df: pd.DataFrame, prompt_type: str = "standard") -> pd.DataFrame:
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating [{prompt_type}]"):
        result = verify_claim(
            claim=row["content"],
            book_name=row["book_name"],
            prompt_type=prompt_type,
        )
        rows.append({
            "id":           row["id"],
            "book_name":    row["book_name"],
            "claim":        row["content"],
            "ground_truth": row["ground_truth"],
            "true_label":   row["label"],
            "predicted":    result["verdict"],
            "correct":      int(result["verdict"] == row["ground_truth"]),
            "raw_response": result["raw_response"],
            "context":      result["context"],
        })
    return pd.DataFrame(rows)


# Run the standard (baseline) evaluation
print("Running baseline evaluation... (this may take a few minutes)")
baseline_results = run_evaluation(train_df, prompt_type="standard")
print("Done.")
# ─── Print summary ───
overall_acc = baseline_results["correct"].mean()
print(f"\nOverall Accuracy: {overall_acc:.1%}")
print("\nAccuracy by book:")
print(baseline_results.groupby("book_name")["correct"].mean().apply(lambda x: f"{x:.1%}").to_string())
print("\nAccuracy by label:")
print(baseline_results.groupby("true_label")["correct"].mean().apply(lambda x: f"{x:.1%}").to_string())

Running baseline evaluation... (this may take a few minutes)


Evaluating [standard]:   9%|▉         | 7/80 [00:17<03:07,  2.56s/it][transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Evaluating [standard]: 100%|██████████| 80/80 [03:36<00:00,  2.71s/it]

Done.

Overall Accuracy: 58.8%

Accuracy by book:
book_name
In Search of the Castaways    61.2%
The Count of Monte Cristo     54.8%

Accuracy by label:
true_label
consistent    64.7%
contradict    48.3%


In [ ]:
# ─────────────────────────────────────────────
# 3C: Compute and display accuracy metrics
# ─────────────────────────────────────────────

def print_accuracy_report(results_df: pd.DataFrame, label: str = "Baseline"):
    """
    Prints a formatted accuracy report.
    """
    total = len(results_df)
    # Exclude undecided (-1) from accuracy calculation
    decided = results_df[results_df['predicted'] != -1]
    undecided = total - len(decided)

    correct = decided['correct'].sum()
    accuracy = correct / len(decided) * 100 if len(decided) > 0 else 0

    # Per-class accuracy
    consistent_cases = decided[decided['ground_truth'] == 1]
    inconsistent_cases = decided[decided['ground_truth'] == 0]
    consistent_acc = consistent_cases['correct'].mean() * 100 if len(consistent_cases) > 0 else 0
    inconsistent_acc = inconsistent_cases['correct'].mean() * 100 if len(inconsistent_cases) > 0 else 0

    print(f"\n{'='*55}")
    print(f"  ACCURACY REPORT — {label}")
    print(f"{'='*55}")
    print(f"  Total test cases:         {total}")
    print(f"  Undecided (parse failed): {undecided}")
    print(f"  Overall accuracy:         {accuracy:.1f}%  ({int(correct)}/{len(decided)})")
    print(f"  Accuracy on CONSISTENT:   {consistent_acc:.1f}%")
    print(f"  Accuracy on INCONSISTENT: {inconsistent_acc:.1f}%")
    print(f"{'='*55}\n")

    # Show failures
    failures = results_df[results_df['correct'] == 0]
    for _, row in failures.iterrows():
        print(f"  ✗ Claim: {row['claim'][:80]}...")
        print(f"    Truth: {row['true_label']} | Predicted: {row['predicted']}")
        print()
    if len(failures) > 0:
        print("  FAILURES (cases where the system was wrong):")
        for _, row in failures.iterrows():
            verdict_label = {1: "CONSISTENT", 0: "INCONSISTENT", -1: "UNDECIDED"}
            print(f"    ✗ [{verdict_label.get(row['ground_truth'])} → predicted {verdict_label.get(row['predicted'])}]")
            print(f"      Claim: {row['claim'][:80]}...")
            #print(f"      Note:  {row['note']}")
            print()


print_accuracy_report(baseline_results, label="Standard Prompt (Baseline)")
baseline_results


  ACCURACY REPORT — Standard Prompt (Baseline)
  Total test cases:         80
  Undecided (parse failed): 0
  Overall accuracy:         58.8%  (47/80)
  Accuracy on CONSISTENT:   64.7%
  Accuracy on INCONSISTENT: 48.3%

  ✗ Claim: Before each fight he studied the crack-patterns of his mother’s shark-tooth neck...
    Truth: consistent | Predicted: 0

  ✗ Claim: Villefort’s drift toward the royalists disappointed him; father and son argued p...
    Truth: contradict | Predicted: 1

  ✗ Claim: The mutiny began when Captain Grant uncovered his forged logbook and threatened ...
    Truth: contradict | Predicted: 1

  ✗ Claim: He rescued the indigenous elder Yurook from colonists and gained tribal protecti...
    Truth: consistent | Predicted: 0

  ✗ Claim: In a skirmish at a British outpost friendly fire killed several of his comrades;...
    Truth: consistent | Predicted: 0

  ✗ Claim: Through underground circles he met the Count of Monte Cristo and fed the avenger...
    Truth: contradi

,id,book_name,claim,ground_truth,true_label,predicted,correct,raw_response,context
0,46,In Search of the Castaways,Thalcave’s people faded as colonists advanced;...,1,consistent,1,1,1\nThe answer is: 1\nThe claim is consistent w...,"[""Chiefs of tribes that were very powerful thi..."
1,137,The Count of Monte Cristo,"Suspected again in 1815, he was re-arrested an...",0,contradict,0,1,0\nThe reasoning: The claim states that the ch...,"[“This will do,” said he, “and from this lette..."
2,74,In Search of the Castaways,Before each fight he studied the crack-pattern...,1,consistent,0,0,0\nThe claim is inconsistent with the story pa...,[in a rigorously straight line. As he advanced...
3,109,The Count of Monte Cristo,Villefort’s drift toward the royalists disappo...,0,contradict,1,0,1\nAnswer: 0\nThe answer is 0.\nExplanation: T...,"[“Madame,” replied Villefort, with a mournful ..."
4,104,The Count of Monte Cristo,His parents were targeted in a reprisal for su...,1,consistent,1,1,1\nAnswer: 0\nThe answer is 0.\nThe answer is ...,"[“But,” said Morrel, “the culprit—the murderer..."
...,...,...,...,...,...,...,...,...,...
75,90,The Count of Monte Cristo,To obtain royalist intelligence from the Vendé...,1,consistent,0,0,0\nThe claim is inconsistent with the story pa...,"[“Ah, madame,” replied Monte Cristo, “all this..."
76,100,The Count of Monte Cristo,Growing up in Paris he devoured Voltaire and R...,1,consistent,1,1,1\nAnswer: 0\nAnswer: 1\nAnswer: 0\nAnswer: 1\...,"[Thus Dantès, who but three months before had ..."
77,138,The Count of Monte Cristo,Long political warfare severed him from his fa...,1,consistent,1,1,"1\nThe answer is: 1\n\nThe claim states that ""...",[most sanguine looked upon any attempt of Napo...
78,130,The Count of Monte Cristo,What seemed an epileptic fit was in fact sudde...,0,contradict,1,0,1\nThe reasoning behind this is that the passa...,[and then his mind was made up—when the jailer...


In [ ]:
# Manual retrieval diagnosis — no RAGAS needed
results = []
for _, row in baseline_results[baseline_results["correct"] == 0].iterrows():
    chunks = " ".join(row["context"]).lower()
    claim  = row["claim"].lower()

    # Check if key words from the claim appear in retrieved chunks
    claim_words = [w for w in claim.split() if len(w) > 4]
    hits = sum(1 for w in claim_words if w in chunks)
    coverage = hits / len(claim_words) if claim_words else 0

    failure_type = "retrieval" if coverage < 0.3 else "reasoning"

    results.append({
        "claim":        row["claim"][:80],
        "true_label":   row["true_label"],
        "coverage":     round(coverage, 2),
        "failure_type": failure_type,
        "book":         row["book_name"],
    })

failures_df = pd.DataFrame(results)
print(failures_df.to_string(index=False))
print(f"\nRetrieval failures: {(failures_df['failure_type'] == 'retrieval').sum()}")
print(f"Reasoning failures: {(failures_df['failure_type'] == 'reasoning').sum()}")

In [ ]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# COMPREHENSIVE EVALUATION REPORT
# ─────────────────────────────────────────────

# ── 1. Overall Accuracy ──────────────────────
total    = len(baseline_results)
correct  = baseline_results["correct"].sum()
accuracy = correct / total

print("=" * 65)
print("  SECTION 1: OVERALL ACCURACY")
print("=" * 65)
print(f"  Total cases:   {total}")
print(f"  Correct:       {correct}")
print(f"  Wrong:         {total - correct}")
print(f"  Accuracy:      {accuracy:.1%}")

# ── 2. Accuracy by Label ─────────────────────
print("\n" + "=" * 65)
print("  SECTION 2: ACCURACY BY LABEL")
print("=" * 65)
for label in ["consistent", "contradict"]:
    grp = baseline_results[baseline_results["true_label"] == label]
    acc = grp["correct"].mean()
    print(f"  {label:<12}  {acc:.1%}  ({grp['correct'].sum()}/{len(grp)})")

# ── 3. Accuracy by Book ──────────────────────
print("\n" + "=" * 65)
print("  SECTION 3: ACCURACY BY BOOK")
print("=" * 65)
for book, grp in baseline_results.groupby("book_name"):
    acc = grp["correct"].mean()
    print(f"  {book[:40]:<40}  {acc:.1%}  ({grp['correct'].sum()}/{len(grp)})")

# ── 4. Accuracy by Book AND Label ────────────
print("\n" + "=" * 65)
print("  SECTION 4: ACCURACY BY BOOK × LABEL")
print("=" * 65)
pivot = baseline_results.groupby(["book_name", "true_label"])["correct"].mean().unstack()
print(pivot.map(lambda x: f"{x:.1%}").to_string())

# ── 5. Prediction Distribution ───────────────
print("\n" + "=" * 65)
print("  SECTION 5: PREDICTION DISTRIBUTION")
print("=" * 65)
pred_counts = baseline_results["predicted"].map({1: "consistent", 0: "contradict", -1: "undecided"}).value_counts()
for label, count in pred_counts.items():
    pct = count / total
    bar = "█" * int(pct * 30)
    print(f"  {label:<12} {count:>3}  {pct:.1%}  {bar}")

# ── 6. Confusion Matrix ──────────────────────
print("\n" + "=" * 65)
print("  SECTION 6: CONFUSION MATRIX")
print("=" * 65)
tp = ((baseline_results["predicted"] == 1) & (baseline_results["ground_truth"] == 1)).sum()
tn = ((baseline_results["predicted"] == 0) & (baseline_results["ground_truth"] == 0)).sum()
fp = ((baseline_results["predicted"] == 1) & (baseline_results["ground_truth"] == 0)).sum()
fn = ((baseline_results["predicted"] == 0) & (baseline_results["ground_truth"] == 1)).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"  {'':20} Predicted consistent  Predicted contradict")
print(f"  {'True consistent':20} {tp:>20} {fn:>20}")
print(f"  {'True contradict':20} {fp:>20} {tn:>20}")
print()
print(f"  Precision: {precision:.3f}  (of cases called consistent, how many were right)")
print(f"  Recall:    {recall:.3f}  (of actually consistent cases, how many did we catch)")
print(f"  F1 Score:  {f1:.3f}")

# ── 7. Failure Analysis ──────────────────────
print("\n" + "=" * 65)
print("  SECTION 7: FAILURE ANALYSIS (retrieval vs reasoning)")
print("=" * 65)

failure_rows = []
for _, row in baseline_results[baseline_results["correct"] == 0].iterrows():
    chunks     = " ".join(row["context"]).lower() if isinstance(row["context"], list) else ""
    claim      = row["claim"].lower()
    claim_words = [w.strip(".,!?") for w in claim.split() if len(w) > 4]
    hits        = sum(1 for w in claim_words if w in chunks)
    coverage    = hits / len(claim_words) if claim_words else 0
    failure_type = "retrieval" if coverage < 0.3 else "reasoning"

    failure_rows.append({
        "book":         row["book_name"].split()[-1],   # short name
        "true_label":   row["true_label"],
        "predicted":    "consistent" if row["predicted"] == 1 else "contradict",
        "coverage":     round(coverage, 2),
        "failure_type": failure_type,
        "claim":        row["claim"][:70],
    })

failures_df = pd.DataFrame(failure_rows)

retrieval_count = (failures_df["failure_type"] == "retrieval").sum()
reasoning_count = (failures_df["failure_type"] == "reasoning").sum()
print(f"  Total failures:    {len(failures_df)}")
print(f"  Retrieval failures: {retrieval_count}  (wrong chunks returned)")
print(f"  Reasoning failures: {reasoning_count}  (right chunks, wrong conclusion)")
print()
print(failures_df[["book", "true_label", "predicted", "coverage", "failure_type", "claim"]]
      .sort_values("failure_type")
      .to_string(index=False))

# ── 8. Failure breakdown by book and type ────
print("\n" + "=" * 65)
print("  SECTION 8: FAILURE TYPE BY BOOK")
print("=" * 65)
print(failures_df.groupby(["book", "failure_type"]).size().unstack(fill_value=0).to_string())

# ── 9. Context coverage distribution ─────────
print("\n" + "=" * 65)
print("  SECTION 9: CONTEXT COVERAGE DISTRIBUTION (failures only)")
print("=" * 65)
bins   = [0, 0.2, 0.4, 0.6, 0.8, 1.01]
labels = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]
failures_df["coverage_bin"] = pd.cut(failures_df["coverage"], bins=bins, labels=labels, right=False)
print(failures_df["coverage_bin"].value_counts().sort_index().to_string())
print()
print("  Low coverage (< 0.3) = retrieval likely failed")
print("  High coverage (> 0.6) = model had the info but reasoned incorrectly")

# ── 10. Summary for written analysis ─────────
print("\n" + "=" * 65)
print("  SECTION 10: KEY FINDINGS FOR WRITTEN ANALYSIS")
print("=" * 65)
dominant_failure = "retrieval" if retrieval_count > reasoning_count else "reasoning"
weak_book        = baseline_results.groupby("book_name")["correct"].mean().idxmin().split()[-1]
weak_label       = baseline_results.groupby("true_label")["correct"].mean().idxmin()
print(f"  - Dominant failure type:       {dominant_failure}")
print(f"  - Weakest book:                {weak_book}")
print(f"  - Weakest label:               {weak_label}")
print(f"  - F1 score:                    {f1:.3f}")
print(f"  - Precision:                   {precision:.3f}")
print(f"  - Recall:                      {recall:.3f}")
print(f"  - % cases with low retrieval:  {retrieval_count/len(failures_df):.1%}")

  SECTION 1: OVERALL ACCURACY
  Total cases:   80
  Correct:       47
  Wrong:         33
  Accuracy:      58.8%

  SECTION 2: ACCURACY BY LABEL
  consistent    64.7%  (33/51)
  contradict    48.3%  (14/29)

  SECTION 3: ACCURACY BY BOOK
  In Search of the Castaways                61.2%  (30/49)
  The Count of Monte Cristo                 54.8%  (17/31)

  SECTION 4: ACCURACY BY BOOK × LABEL
true_label                 consistent contradict
book_name                                       
In Search of the Castaways      55.6%      76.9%
The Count of Monte Cristo       86.7%      25.0%

  SECTION 5: PREDICTION DISTRIBUTION
  consistent    48  60.0%  ██████████████████
  contradict    32  40.0%  ████████████

  SECTION 6: CONFUSION MATRIX
                       Predicted consistent  Predicted contradict
  True consistent                        33                   18
  True contradict                        15                   14

  Precision: 0.688  (of cases called consistent, how many

In [ ]:
# ─────────────────────────────────────────────
# RAGAS 1: Install
# ─────────────────────────────────────────────
!pip install -q ragas datasets

In [ ]:
from ragas import evaluate
from ragas.metrics.collections import ContextRecall, ContextPrecision
from ragas.llms import LangchainLLMWrapper
from langchain_community.llms import HuggingFacePipeline

# Wrap your existing llm_pipeline for RAGAS
ragas_llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=llm_pipeline))

ragas_scores = evaluate(
    ragas_data,
    metrics=[
        ContextRecall(llm=ragas_llm),
        ContextPrecision(llm=ragas_llm)
    ]
)
print(ragas_scores)

In [ ]:
# Per-row scores — use to classify retrieval vs reasoning failures
# Must use ragas_scores.to_pandas(), not ragas_data.to_pandas()
scores_df = ragas_scores.to_pandas()
print(scores_df.columns.tolist())  # check exact column names first
scores_df["true_label"]  = baseline_results["true_label"].values
scores_df["correct"]     = baseline_results["correct"].values
scores_df["book_name"]   = baseline_results["book_name"].values

failures = scores_df[scores_df["correct"] == 0].copy()

# Classify failure type
# Low context_recall = retrieval failure (right chunks never came back)
# High context_recall but low faithfulness = reasoning failure (chunks were there but model ignored them)
def classify_failure(row):
    if row["context_recall"] < 0.5:
        return "retrieval"
    else:
        return "reasoning"

failures["failure_type"] = failures.apply(classify_failure, axis=1)

print(failures[["question", "true_label", "context_recall", "faithfulness", "failure_type"]]
      .to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# RAGAS 4: Print comparison report
# ─────────────────────────────────────────────

print("\n" + "=" * 55)
print("  RAGAS EVALUATION REPORT")
print("=" * 55)
print(f"  {'Metric':<25} {'Standard':>10} {'CoT':>10} {'Δ':>8}")
print("-" * 55)

for metric in ["faithfulness", "answer_relevancy", "context_recall", "context_precision"]:
    b = ragas_baseline_scores[metric]
    c = ragas_cot_scores[metric]
    print(f"  {metric:<25} {b:>10.3f} {c:>10.3f} {c-b:>+8.3f}")

print("=" * 55)
print()
print("What each metric means for your task:")
print("  faithfulness      — do your verdicts match what the retrieved chunks say?")
print("  answer_relevancy  — is the verdict actually answering the claim?")
print("  context_recall    — did retrieval surface the chunks needed to answer?")
print("  context_precision — were the retrieved chunks relevant, or mostly noise?")

# Also export as DataFrame for your report
ragas_df = pd.DataFrame({
    "metric":   ["faithfulness", "answer_relevancy", "context_recall", "context_precision"],
    "standard": [ragas_baseline_scores[m] for m in ["faithfulness", "answer_relevancy", "context_recall", "context_precision"]],
    "cot":      [ragas_cot_scores[m]      for m in ["faithfulness", "answer_relevancy", "context_recall", "context_precision"]],
})
ragas_df["delta"] = ragas_df["cot"] - ragas_df["standard"]
print()
print(ragas_df.to_string(index=False))

In [ ]:
@title
# ─────────────────────────────────────────────
# RAGAS 5: Per-row faithfulness (find least faithful predictions)
# Useful for diagnosing reasoning failures
# ─────────────────────────────────────────────

ragas_baseline_df = ragas_baseline_scores.to_pandas()
ragas_baseline_df["true_label"]      = baseline_results["true_label"].values

ragas_baseline_df["predicted_label"] = baseline_results["predicted_label"].values
ragas_baseline_df["correct"]         = baseline_results["correct"].values
ragas_baseline_df["book_name"]       = baseline_results["book_name"].values

# Show least faithful predictions — these are likely reasoning failures
low_faith = ragas_baseline_df.sort_values("faithfulness").head(10)
print("10 least faithful predictions (likely reasoning failures):")
print(low_faith[["question", "faithfulness", "predicted_label", "true_label", "correct", "book_name"]]
      .to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# HYBRID SEARCH SETUP
# ─────────────────────────────────────────────
!pip install -q rank_bm25 sentence-transformers

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
import numpy as np

# ── Build BM25 index for each book ───────────
def tokenize(text):
    return text.lower().split()

bm25_indexes = {}
bm25_chunks  = {}

for book_name, chunks in all_chunks.items():
    tokenized = [tokenize(c) for c in chunks]
    bm25_indexes[book_name] = BM25Okapi(tokenized)
    bm25_chunks[book_name]  = chunks
    print(f"BM25 index built for '{book_name}': {len(chunks)} chunks")


# ── Reciprocal Rank Fusion ────────────────────
def reciprocal_rank_fusion(rankings: list[list[str]], k: int = 60) -> list[str]:
    """
    Takes multiple ranked lists of chunks and merges them using RRF.
    score(chunk) = sum of 1/(rank + k) across all lists.
    """
    scores = {}
    for ranked_list in rankings:
        for rank, chunk in enumerate(ranked_list):
            scores[chunk] = scores.get(chunk, 0) + 1 / (rank + k)
    return sorted(scores, key=scores.get, reverse=True)


# ── Hybrid retriever ──────────────────────────
def retrieve_context_hybrid(query: str, book_name: str, k: int = 3,
                             use_reranker: bool = False) -> list[str]:
    # ── Vector search results ──
    vector_results = vector_dbs[book_name].similarity_search(query, k=k * 5)
    vector_ranked  = [doc.page_content for doc in vector_results]

    # ── BM25 keyword search results ──
    tokenized_query = tokenize(query)
    bm25_scores     = bm25_indexes[book_name].get_scores(tokenized_query)
    top_bm25_idx    = np.argsort(bm25_scores)[::-1][:k * 5]
    bm25_ranked     = [bm25_chunks[book_name][i] for i in top_bm25_idx]

    # ── Merge with RRF ──
    merged = reciprocal_rank_fusion([vector_ranked, bm25_ranked])

    # ── Deduplicate ──
    seen   = set()
    unique = []
    for chunk in merged:
        content = " ".join(chunk.split())
        if content not in seen:
            seen.add(content)
            unique.append(chunk)
        if len(unique) == (k * 3 if use_reranker else k):
            break

    # ── Optional Cross-Encoder reranking ──
    if use_reranker and len(unique) > k:
        reranker   = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        pairs      = [[query, chunk] for chunk in unique]
        re_scores  = reranker.predict(pairs)
        ranked_idx = np.argsort(re_scores)[::-1][:k]
        unique     = [unique[i] for i in ranked_idx]

    return unique[:k]


# ── Quick sanity check ────────────────────────
sample = train_df.iloc[1]
hybrid_chunks = retrieve_context_hybrid(sample["content"], book_name=sample["book_name"])
print(f"Claim: {sample['content'][:80]}...")
print(f"Book:  {sample['book_name']}\n")
for i, c in enumerate(hybrid_chunks):
    print(f"--- Chunk {i+1} ---\n{c[:200]}\n")

BM25 index built for 'The Count of Monte Cristo': 9646 chunks
BM25 index built for 'In Search of the Castaways': 3068 chunks
Claim: Suspected again in 1815, he was re-arrested and shipped to the Château d’If, thi...
Book:  The Count of Monte Cristo

--- Chunk 1 ---
never-to-be-forgotten night of his departure for the Château d’If, he
had been put on board the boat destined to convey him thither.

--- Chunk 2 ---
had been thrown alive from the top of the Château d’If, and that the
cry you uttered as you dashed upon the rocks first revealed to your
jailers that they were your murderers. Well, Edmond, I swear to

--- Chunk 3 ---
“This will do,” said he, “and from this letter, which might have ruined
me, I will make my fortune. Now to the work I have in hand.” And after
having assured himself that the prisoner was gone, the de



In [ ]:
# ─────────────────────────────────────────────
# HYBRID EVALUATION + COMPARISON
# ─────────────────────────────────────────────

def run_evaluation_hybrid(df: pd.DataFrame, use_reranker: bool = False) -> pd.DataFrame:
    rows = []
    label = "hybrid+reranker" if use_reranker else "hybrid"
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating [{label}]"):
        chunks = retrieve_context_hybrid(
            row["content"],
            book_name=row["book_name"],
            use_reranker=use_reranker,
        )
        prompt      = build_standard_prompt(row["content"], chunks)
        raw         = call_llm(prompt)
        verdict     = parse_verdict(raw)
        rows.append({
            "id":           row["id"],
            "book_name":    row["book_name"],
            "claim":        row["content"],
            "ground_truth": row["ground_truth"],
            "true_label":   row["label"],
            "predicted":    verdict,
            "correct":      int(verdict == row["ground_truth"]),
            "raw_response": raw,
            "context":      chunks,
        })
    return pd.DataFrame(rows)


# Run hybrid evaluation
print("Running hybrid evaluation...")
hybrid_results = run_evaluation_hybrid(train_df, use_reranker=False)

# ── Side by side comparison ───────────────────
print("\n" + "=" * 65)
print("  RETRIEVAL COMPARISON: Vector vs Hybrid")
print("=" * 65)

for label, df in [("Vector (baseline)", baseline_results), ("Hybrid (BM25 + Vector)", hybrid_results)]:
    acc = df["correct"].mean()
    con = df[df["true_label"] == "consistent"]["correct"].mean()
    ctr = df[df["true_label"] == "contradict"]["correct"].mean()
    print(f"\n  {label}")
    print(f"    Overall:     {acc:.1%}")
    print(f"    Consistent:  {con:.1%}")
    print(f"    Contradict:  {ctr:.1%}")

# ── Retrieval failure comparison ──────────────
print("\n" + "=" * 65)
print("  RETRIEVAL FAILURE COMPARISON")
print("=" * 65)

def count_retrieval_failures(results_df):
    count = 0
    for _, row in results_df[results_df["correct"] == 0].iterrows():
        chunks      = " ".join(row["context"]).lower() if isinstance(row["context"], list) else ""
        claim_words = [w.strip(".,!?") for w in row["claim"].lower().split() if len(w) > 4]
        hits        = sum(1 for w in claim_words if w in chunks)
        coverage    = hits / len(claim_words) if claim_words else 0
        if coverage < 0.3:
            count += 1
    return count

baseline_retrieval_failures = count_retrieval_failures(baseline_results)
hybrid_retrieval_failures   = count_retrieval_failures(hybrid_results)
fixed = baseline_retrieval_failures - hybrid_retrieval_failures

print(f"  Baseline retrieval failures: {baseline_retrieval_failures}")
print(f"  Hybrid retrieval failures:   {hybrid_retrieval_failures}")
print(f"  Fixed by hybrid search:      {fixed}")
print(f"  Accuracy change:             {hybrid_results['correct'].mean() - baseline_results['correct'].mean():+.1%}")

Running hybrid evaluation...


Evaluating [hybrid]:   0%|          | 0/80 [00:00<?, ?it/s][transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Evaluating [hybrid]: 100%|██████████| 80/80 [04:19<00:00,  3.25s/it]


  RETRIEVAL COMPARISON: Vector vs Hybrid


NameError: name 'baseline_results' is not defined

### Analysis — Why Did the Baseline Fail?

**TODO: Fill this in after running the evaluation above.**

Look at the failures from the report and answer:

1. **What types of claims did the baseline get wrong?** (Numerical? Negation? Implied facts? Cases where the relevant chunk wasn't retrieved?)

2. **Was retrieval the problem, or reasoning?** (You can check by printing `result['context']` for failing cases — did the right chunk even get retrieved?)

3. **What is your hypothesis for why it failed?** (e.g., "The model saw the word 'parents' in both the claim and a chunk, and assumed they matched without reasoning about the difference between having parents vs. not having them.")

> Write your analysis here as plain text. 3–5 sentences is enough.

My analysis
one point of analysis is that the model predicted consistent claims correctly 80% of the time but inconsistent claims correctly only 50% of the time. so its false positive rate is quite high. It also got one location contradiction and two issues related to names. Also in the harder claims it failed to connnect context from early and late chapters and also the overarching plot of the story, indicating the context retrieval is not broad enough.

---
## Section 4: Chain-of-Thought Prompting
**Week 6 Goal:** Instead of asking the model to jump straight to a verdict, force it to *reason step by step* before answering. This is called **Chain-of-Thought (CoT) prompting**.

### Why Does This Work?
A standard prompt asks: *"Is this consistent? Yes or No."* The model pattern-matches and often gets it wrong on subtle cases.

A CoT prompt asks: *"Step 1: What does the story say? Step 2: What does the claim say? Step 3: Are they the same?"* The model must read carefully before concluding.

Think of it like the difference between asking a student *"Is the answer 4?"* vs. *"Show your working."*

In [ ]:
# ─────────────────────────────────────────────
# 4A: Build the Chain-of-Thought Prompt
# ─────────────────────────────────────────────

def build_cot_prompt(claim: str, context_chunks: list[str]) -> str:
    """
    Builds a Chain-of-Thought prompt that forces the model to reason
    step-by-step before giving a verdict.

    Args:
        claim: The backstory claim to verify.
        context_chunks: Retrieved story passages.

    Returns:
        A formatted CoT prompt string.
    """
    context_str = "\n\n".join([f"[Passage {i+1}]: {chunk}" for i, chunk in enumerate(context_chunks)])

    prompt = f"""You are a story fact-checker. You must reason carefully before giving a verdict.

STORY PASSAGES:
{context_str}

CLAIM TO VERIFY: "{claim}"

Think through this step by step:

Step 1 — What does the story say about the relevant character or event? Summarize in one sentence.
Step 2 — What does the CLAIM say about the same character or event? Summarize in one sentence.
Step 3 — Are Step 1 and Step 2 saying the same thing, or do they contradict each other?
Step 4 — Final verdict: Answer with only '1' if consistent, or '0' if inconsistent.

Step 1:"""

    return prompt

In [ ]:
# ─────────────────────────────────────────────
# 4B: Update parse_verdict to handle CoT output
# ─────────────────────────────────────────────
# With CoT, the model outputs a reasoning chain THEN a final answer.
# The verdict is always in the last part of the response.
# We need a smarter parser that looks at the end of the response.

def parse_verdict_cot(llm_response: str) -> int:
    """
    Parses the verdict from a CoT response by searching from the end.
    The final step's verdict is more reliable than the beginning.
    """
    # TODO: Improve this parser for your specific model's output format.
    # Hint: The CoT model might say "Step 4: 1" or "Final answer: Consistent" or just "1".
    # Consider searching backwards through the response for the last occurrence of 0 or 1.

    response_lower = llm_response.lower()

    # Search from the end for a verdict signal
    if 'step 4' in response_lower:
        # Extract everything after Step 4
        after_step4 = response_lower.split('step 4')[-1]
        if '1' in after_step4 and 'inconsistent' not in after_step4:
            return 1
        elif '0' in after_step4 or 'inconsistent' in after_step4:
            return 0

    # Fallback to standard parser
    return parse_verdict(llm_response)



In [ ]:
def verify_claim(claim: str, book_name: str, k: int = 3, prompt_type: str = "standard", verbose: bool = False) -> dict:
    context_chunks = retrieve_context(claim, book_name=book_name, k=k)

    if prompt_type == "cot":
        prompt = build_cot_prompt(claim, context_chunks)
        max_tokens = 250
    else:
        prompt = build_standard_prompt(claim, context_chunks)
        max_tokens = 50

    if verbose:
        print("=" * 60)
        print(f"[{prompt_type.upper()}] CLAIM: {claim}")
        print("-" * 60)

    raw_response = call_llm(prompt, max_new_tokens=max_tokens)

    if verbose:
        print(f"RAW RESPONSE:\n{raw_response}")

    if prompt_type == "cot":
        verdict = parse_verdict_cot(raw_response)
    else:
        verdict = parse_verdict(raw_response)

    if verbose:
        verdict_label = {1: "CONSISTENT ✅", 0: "INCONSISTENT ❌", -1: "UNDECIDED ⚠️"}
        print(f"\nVERDICT: {verdict_label.get(verdict, '?')}")
        print("=" * 60)

    return {
        "claim":        claim,
        "verdict":      verdict,
        "raw_response": raw_response,
        "context":      context_chunks
    }

print("verify_claim updated with CoT support.")

verify_claim updated with CoT support.


---
## Section 5: Side-by-Side Comparison
Show examples where the standard prompt failed but the CoT prompt succeeded.

In [ ]:
# ─────────────────────────────────────────────
# 5A: Run CoT evaluation on the same dataset
# ─────────────────────────────────────────────

print("Running CoT evaluation... (this will take longer due to longer outputs)")
cot_results = run_evaluation(train_df, prompt_type="cot")
print("Done.")

Running CoT evaluation... (this will take longer due to longer outputs)


Evaluating [cot]: 100%|██████████| 80/80 [18:23<00:00, 13.80s/it]

Done.


In [ ]:
# ─────────────────────────────────────────────
# 5B: Compare Standard vs CoT accuracy
# ─────────────────────────────────────────────

print_accuracy_report(baseline_results, label="Standard Prompt (Baseline)")
print_accuracy_report(cot_results, label="Chain-of-Thought Prompt")

# Summary comparison
baseline_acc = baseline_results[baseline_results['predicted'] != -1]['correct'].mean() * 100
cot_acc = cot_results[cot_results['predicted'] != -1]['correct'].mean() * 100
print(f"\nACCURACY IMPROVEMENT: {cot_acc - baseline_acc:+.1f} percentage points")


  ACCURACY REPORT — Standard Prompt (Baseline)
  Total test cases:         80
  Undecided (parse failed): 0
  Overall accuracy:         60.0%  (48/80)
  Accuracy on CONSISTENT:   66.7%
  Accuracy on INCONSISTENT: 48.3%

  ✗ Claim: Before each fight he studied the crack-patterns of his mother’s shark-tooth neck...
    Truth: consistent | Predicted: 0

  ✗ Claim: Villefort’s drift toward the royalists disappointed him; father and son argued p...
    Truth: contradict | Predicted: 1

  ✗ Claim: The mutiny began when Captain Grant uncovered his forged logbook and threatened ...
    Truth: contradict | Predicted: 1

  ✗ Claim: He rescued the indigenous elder Yurook from colonists and gained tribal protecti...
    Truth: consistent | Predicted: 0

  ✗ Claim: In a skirmish at a British outpost friendly fire killed several of his comrades;...
    Truth: consistent | Predicted: 0

  ✗ Claim: Through underground circles he met the Count of Monte Cristo and fed the avenger...
    Truth: contradi

In [ ]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# COMPREHENSIVE EVALUATION REPORT — CoT
# ─────────────────────────────────────────────

# ── 1. Overall Accuracy ──────────────────────
total    = len(cot_results)
correct  = cot_results["correct"].sum()
accuracy = correct / total

print("=" * 65)
print("  SECTION 1: OVERALL ACCURACY")
print("=" * 65)
print(f"  Total cases:   {total}")
print(f"  Correct:       {correct}")
print(f"  Wrong:         {total - correct}")
print(f"  Accuracy:      {accuracy:.1%}")

# ── 2. Accuracy by Label ─────────────────────
print("\n" + "=" * 65)
print("  SECTION 2: ACCURACY BY LABEL")
print("=" * 65)
for label in ["consistent", "contradict"]:
    grp = cot_results[cot_results["true_label"] == label]
    acc = grp["correct"].mean()
    print(f"  {label:<12}  {acc:.1%}  ({grp['correct'].sum()}/{len(grp)})")

# ── 3. Accuracy by Book ──────────────────────
print("\n" + "=" * 65)
print("  SECTION 3: ACCURACY BY BOOK")
print("=" * 65)
for book, grp in cot_results.groupby("book_name"):
    acc = grp["correct"].mean()
    print(f"  {book[:40]:<40}  {acc:.1%}  ({grp['correct'].sum()}/{len(grp)})")

# ── 4. Accuracy by Book AND Label ────────────
print("\n" + "=" * 65)
print("  SECTION 4: ACCURACY BY BOOK × LABEL")
print("=" * 65)
pivot = cot_results.groupby(["book_name", "true_label"])["correct"].mean().unstack()
print(pivot.map(lambda x: f"{x:.1%}" if pd.notna(x) else "N/A").to_string())

# ── 5. Prediction Distribution ───────────────
print("\n" + "=" * 65)
print("  SECTION 5: PREDICTION DISTRIBUTION")
print("=" * 65)
pred_counts = cot_results["predicted"].map({1: "consistent", 0: "contradict", -1: "undecided"}).value_counts()
for label, count in pred_counts.items():
    pct = count / total
    bar = "█" * int(pct * 30)
    print(f"  {label:<12} {count:>3}  {pct:.1%}  {bar}")

# ── 6. Confusion Matrix ──────────────────────
print("\n" + "=" * 65)
print("  SECTION 6: CONFUSION MATRIX")
print("=" * 65)
tp = ((cot_results["predicted"] == 1) & (cot_results["ground_truth"] == 1)).sum()
tn = ((cot_results["predicted"] == 0) & (cot_results["ground_truth"] == 0)).sum()
fp = ((cot_results["predicted"] == 1) & (cot_results["ground_truth"] == 0)).sum()
fn = ((cot_results["predicted"] == 0) & (cot_results["ground_truth"] == 1)).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"  {'':20} Predicted consistent  Predicted contradict")
print(f"  {'True consistent':20} {tp:>20} {fn:>20}")
print(f"  {'True contradict':20} {fp:>20} {tn:>20}")
print()
print(f"  Precision: {precision:.3f}  (of cases called consistent, how many were right)")
print(f"  Recall:    {recall:.3f}  (of actually consistent cases, how many did we catch)")
print(f"  F1 Score:  {f1:.3f}")

# ── 7. Failure Analysis ──────────────────────
print("\n" + "=" * 65)
print("  SECTION 7: FAILURE ANALYSIS (retrieval vs reasoning)")
print("=" * 65)

failure_rows = []
for _, row in cot_results[cot_results["correct"] == 0].iterrows():
    chunks      = " ".join(row["context"]).lower() if isinstance(row["context"], list) else ""
    claim       = row["claim"].lower()
    claim_words = [w.strip(".,!?") for w in claim.split() if len(w) > 4]
    hits        = sum(1 for w in claim_words if w in chunks)
    coverage    = hits / len(claim_words) if claim_words else 0
    failure_type = "retrieval" if coverage < 0.3 else "reasoning"

    failure_rows.append({
        "book":         row["book_name"].split()[-1],
        "true_label":   row["true_label"],
        "predicted":    "consistent" if row["predicted"] == 1 else "contradict",
        "coverage":     round(coverage, 2),
        "failure_type": failure_type,
        "claim":        row["claim"][:70],
    })

failures_df = pd.DataFrame(failure_rows)

retrieval_count = (failures_df["failure_type"] == "retrieval").sum()
reasoning_count = (failures_df["failure_type"] == "reasoning").sum()
print(f"  Total failures:    {len(failures_df)}")
print(f"  Retrieval failures: {retrieval_count}  (wrong chunks returned)")
print(f"  Reasoning failures: {reasoning_count}  (right chunks, wrong conclusion)")
print()
print(failures_df[["book", "true_label", "predicted", "coverage", "failure_type", "claim"]]
      .sort_values("failure_type")
      .to_string(index=False))

# ── 8. Failure breakdown by book and type ────
print("\n" + "=" * 65)
print("  SECTION 8: FAILURE TYPE BY BOOK")
print("=" * 65)
print(failures_df.groupby(["book", "failure_type"]).size().unstack(fill_value=0).to_string())

# ── 9. Context coverage distribution ─────────
print("\n" + "=" * 65)
print("  SECTION 9: CONTEXT COVERAGE DISTRIBUTION (failures only)")
print("=" * 65)
bins   = [0, 0.2, 0.4, 0.6, 0.8, 1.01]
labels = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]
failures_df["coverage_bin"] = pd.cut(failures_df["coverage"], bins=bins, labels=labels, right=False)
print(failures_df["coverage_bin"].value_counts().sort_index().to_string())
print()
print("  Low coverage (< 0.3) = retrieval likely failed")
print("  High coverage (> 0.6) = model had the info but reasoned incorrectly")

# ── 10. Summary for written analysis ─────────
print("\n" + "=" * 65)
print("  SECTION 10: KEY FINDINGS FOR WRITTEN ANALYSIS")
print("=" * 65)
dominant_failure = "retrieval" if retrieval_count > reasoning_count else "reasoning"
weak_book        = cot_results.groupby("book_name")["correct"].mean().idxmin().split()[-1]
weak_label       = cot_results.groupby("true_label")["correct"].mean().idxmin()
print(f"  - Dominant failure type:       {dominant_failure}")
print(f"  - Weakest book:                {weak_book}")
print(f"  - Weakest label:               {weak_label}")
print(f"  - F1 score:                    {f1:.3f}")
print(f"  - Precision:                   {precision:.3f}")
print(f"  - Recall:                      {recall:.3f}")
print(f"  - % cases with low retrieval:  {retrieval_count/len(failures_df):.1%}")

  SECTION 1: OVERALL ACCURACY
  Total cases:   80
  Correct:       45
  Wrong:         35
  Accuracy:      56.2%

  SECTION 2: ACCURACY BY LABEL
  consistent    72.5%  (37/51)
  contradict    27.6%  (8/29)

  SECTION 3: ACCURACY BY BOOK
  In Search of the Castaways                59.2%  (29/49)
  The Count of Monte Cristo                 51.6%  (16/31)

  SECTION 4: ACCURACY BY BOOK × LABEL
true_label                 consistent contradict
book_name                                       
In Search of the Castaways      72.2%      23.1%
The Count of Monte Cristo       73.3%      31.2%

  SECTION 5: PREDICTION DISTRIBUTION
  consistent    58  72.5%  █████████████████████
  contradict    21  26.2%  ███████
  undecided      1  1.2%  

  SECTION 6: CONFUSION MATRIX
                       Predicted consistent  Predicted contradict
  True consistent                        37                   13
  True contradict                        21                    8

  Precision: 0.638  (of cases cal

In [ ]:
for _, row in baseline_results[baseline_results["correct"] == 0].iterrows():
    chunks = " ".join(row["context"]) if isinstance(row["context"], list) else ""
    claim = row["claim"].lower()
    claim_words = [w.strip(".,!?") for w in claim.split() if len(w) > 4]
    hits = sum(1 for w in claim_words if w in chunks)
    coverage = hits / len(claim_words) if claim_words else 0

    if coverage >= 0.3:
        print("=" * 65)
        print(f"CLAIM:       {row['claim']}")
        print(f"TRUE LABEL:  {row['true_label']}")
        print(f"PREDICTED:   {row['predicted']}")
        print(f"COVERAGE:    {coverage:.2f}")
        print(f"\nCHUNKS:\n{chunks[:800]}")
        print(f"\nBASELINE RESPONSE:\n{row['raw_response']}")
        print()

In [ ]:
target = "villefort"
for df, label in [(baseline_results, "BASELINE"), (cot_results, "COT")]:
    match = df[df["claim"].str.lower().str.contains(target) & (df["correct"] == 0)]
    for _, row in match.iterrows():
        print(f"[{label}]\nCLAIM: {row['claim']}\nRESPONSE: {row['raw_response']}\n")

[BASELINE]
CLAIM: Villefort’s drift toward the royalists disappointed him; father and son argued politics at every family gathering.
RESPONSE: 1
Answer: 0
The answer is 0.
Explanation: The claim that "Villefort’s drift toward the royalists disappointed him; father and son argued politics at every family gathering" is inconsistent with the story passages.

[BASELINE]
CLAIM: **Father-son Rift**: Discovering that his elder son Gérard (Villefort) sought to wed into the old aristocracy, Noirtier publicly burned the Saint-Méran betrothal contract (1804), forcing Villefort to flee to Paris overnight.
RESPONSE: 1
The reasoning: The claim states that Noirtier burned the betrothal contract, which is mentioned in Passage 2. Passage 2 says that Villefort, after being filled with remorseful memories of Marseilles, sought and

[BASELINE]
CLAIM: At a Vienna-congress salon he briefly watched young prosecutor Villefort tamper with evidence against an Italian revolutionary.
RESPONSE: 1
Answer: 0
Answer:

In [ ]:
# ─────────────────────────────────────────────
# 5D: Also check where CoT REGRESSED (got worse than standard)
# ─────────────────────────────────────────────
# Sometimes CoT can overthink and get simple cases wrong.
# This is an important finding to report.

print("=" * 60)
print("  CASES WHERE CoT REGRESSED (Standard was right, CoT was wrong)")
print("=" * 60)

regressed_cases = []
for i in range(len(EVAL_DATASET)):
    standard_correct = baseline_results.iloc[i]['correct']
    cot_correct = cot_results.iloc[i]['correct']

    if standard_correct == 1 and cot_correct == 0:
        regressed_cases.append(i)
        print(f"  Claim:         {EVAL_DATASET[i]['claim']}")
        print(f"  Note:          {EVAL_DATASET[i]['note']}")
        print()

if not regressed_cases:
    print("No regressions found. CoT improved or matched the baseline on all cases.")

  CASES WHERE CoT REGRESSED (Standard was right, CoT was wrong)


NameError: name 'EVAL_DATASET' is not defined

In [ ]:
# ─────────────────────────────────────────────
# PART 4: AGENT LOOP
# Cell 1: Define the search tool
# ─────────────────────────────────────────────

def search_story(query: str, book_name: str, k: int = 3) -> list[str]:
    """
    Tool the agent can call to retrieve chunks from the story.
    This is just retrieve_context wrapped as a named tool.
    """
    return retrieve_context(query, book_name=book_name, k=k)

In [ ]:
# ─────────────────────────────────────────────
# Cell 2: Confidence check — does the model have enough info?
# ─────────────────────────────────────────────

def build_confidence_prompt(claim: str, context_chunks: list[str]) -> str:
    context_str = "\n\n".join(
        [f"[Passage {i+1}]: {chunk}" for i, chunk in enumerate(context_chunks)]
    )
    return f"""Read these passages and the claim below.

PASSAGES:
{context_str}

CLAIM: "{claim}"

Do the passages contain enough specific information to verify or contradict this claim?
Answer SUFFICIENT if yes, INSUFFICIENT if no.

Answer:"""


def build_refined_query_prompt(claim: str, context_chunks: list[str]) -> str:
    context_str = "\n\n".join(
        [f"[Passage {i+1}]: {chunk}" for i, chunk in enumerate(context_chunks)]
    )
    return f"""You searched for information about this claim but the results were not specific enough.

CLAIM: "{claim}"

RETRIEVED SO FAR:
{context_str}

Write a short, different search query (5-10 words) to find more specific information.
Focus on the most specific name, date, or event in the claim.

Query:"""


def parse_confidence(response: str) -> bool:
    """Returns True if model says SUFFICIENT, False if INSUFFICIENT."""
    text = response.lower().strip()
    if "sufficient" in text and "insufficient" not in text:
        return True
    return False

In [ ]:
# ─────────────────────────────────────────────
# PART 4: AGENT LOOP (fixed)
# ─────────────────────────────────────────────

import re

def extract_refined_query(claim: str, char: str = "") -> str:
    """
    Extracts a refined search query from the claim using Python —
    no LLM needed. Focuses on proper nouns, dates, and numbers
    which are most likely to appear verbatim in the source text.
    """
    # Extract capitalised words (proper nouns)
    proper_nouns = re.findall(r'\b[A-Z][a-z]+\b', claim)
    # Extract years and numbers
    numbers      = re.findall(r'\b\d{4}|\b\d+\b', claim)
    # Build refined query from most specific terms
    terms        = proper_nouns + numbers
    if char:
        terms = [char] + [t for t in terms if t != char]
    # Take up to 6 most specific terms
    query = " ".join(terms[:6])
    return query if query.strip() else claim[:80]


def check_sufficient(chunks: list[str], claim: str) -> bool:
    """
    Check sufficiency in Python — no LLM needed.
    A chunk is considered relevant if it shares key proper nouns with the claim.
    """
    proper_nouns = set(re.findall(r'\b[A-Z][a-z]+\b', claim))
    if not proper_nouns:
        return True  # can't check, proceed

    combined = " ".join(chunks)
    matches  = sum(1 for noun in proper_nouns if noun in combined)
    coverage = matches / len(proper_nouns)

    # Sufficient if at least 40% of proper nouns appear in chunks
    return coverage >= 0.4


def verify_claim_agent(claim: str, book_name: str, char: str = "",
                        max_searches: int = 2, verbose: bool = False) -> dict:
    log            = []
    best_chunks    = []
    best_coverage  = 0
    queries        = [claim]

    if verbose:
        print("=" * 65)
        print(f"AGENT STARTING")
        print(f"CLAIM: {claim[:80]}...")
        print("=" * 65)

    for search_num in range(1, max_searches + 1):
        query      = queries[-1]
        new_chunks = search_story(query, book_name=book_name, k=3)

        # Score this search's coverage
        proper_nouns = set(re.findall(r'\b[A-Z][a-z]+\b', claim))
        combined     = " ".join(new_chunks)
        coverage     = sum(1 for n in proper_nouns if n in combined) / max(len(proper_nouns), 1)

        # Keep whichever search returned better chunks
        if coverage > best_coverage:
            best_coverage = coverage
            best_chunks   = new_chunks

        log.append({
            "step":     search_num,
            "action":   "search",
            "query":    query,
            "chunks":   new_chunks,
            "coverage": round(coverage, 2),
        })

        if verbose:
            print(f"\n[Search {search_num}] Query: '{query[:70]}'")
            print(f"  Coverage: {coverage:.0%}")
            for i, c in enumerate(new_chunks):
                print(f"  Chunk {i+1}: {c[:100]}...")

        is_sufficient = coverage >= 0.4

        log[-1]["sufficient"] = is_sufficient

        if verbose:
            print(f"  Sufficient: {'YES ✓' if is_sufficient else 'NO ✗'}")

        if is_sufficient or search_num == max_searches:
            if not is_sufficient and verbose:
                print(f"\n  [Max searches reached — using best chunks (coverage={best_coverage:.0%})]")
            break

        # Refine query
        refined = extract_refined_query(claim, char=char)
        queries.append(refined)
        log.append({"step": search_num, "action": "refine_query", "query": refined})

        if verbose:
            print(f"  Refined query: '{refined}'")

    # Final verdict using BEST chunks only, not all accumulated
    verdict_prompt = build_standard_prompt(claim, best_chunks)
    raw_response   = call_llm(verdict_prompt, max_new_tokens=50)
    verdict        = parse_verdict(raw_response)

    log.append({
        "step":         search_num + 1,
        "action":       "final_verdict",
        "raw_response": raw_response,
        "verdict":      verdict,
    })

    if verbose:
        label = {1: "CONSISTENT ✅", 0: "INCONSISTENT ❌", -1: "UNDECIDED ⚠️"}
        print(f"\n[Final Verdict] {label.get(verdict, '?')}")
        print(f"Raw: {raw_response.strip()[:80]}")
        print("=" * 65)

    return {
        "claim":         claim,
        "book_name":     book_name,
        "verdict":       verdict,
        "raw_response":  raw_response,
        "context":       best_chunks,
        "searches_used": len(queries),
        "queries":       queries,
        "log":           log,
    }

In [ ]:
# ─────────────────────────────────────────────
# Run agent evaluation
# ─────────────────────────────────────────────

def run_evaluation_agent(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating [agent]"):
        result = verify_claim_agent(
            claim=row["content"],
            book_name=row["book_name"],
            char=row["char"],
            max_searches=2,
            verbose=False,
        )
        rows.append({
            "id":            row["id"],
            "book_name":     row["book_name"],
            "claim":         row["content"],
            "ground_truth":  row["ground_truth"],
            "true_label":    row["label"],
            "predicted":     result["verdict"],
            "correct":       int(result["verdict"] == row["ground_truth"]),
            "raw_response":  result["raw_response"],
            "context":       result["context"],
            "searches_used": result["searches_used"],
            "queries":       result["queries"],
        })
    return pd.DataFrame(rows)


print("Running agent evaluation...")
agent_results = run_evaluation_agent(train_df)
print("Done.")

# Comparison
print("\n" + "=" * 55)
print("  COMPARISON: Baseline vs Agent")
print("=" * 55)
for label, df in [("Baseline", baseline_results), ("Agent", agent_results)]:
    acc = df["correct"].mean()
    con = df[df["true_label"] == "consistent"]["correct"].mean()
    ctr = df[df["true_label"] == "contradict"]["correct"].mean()
    print(f"\n  {label}")
    print(f"    Overall:    {acc:.1%}")
    print(f"    Consistent: {con:.1%}")
    print(f"    Contradict: {ctr:.1%}")

print(f"\n  Avg searches per claim: {agent_results['searches_used'].mean():.2f}")
print(f"  Claims needing 1 search: {(agent_results['searches_used'] == 1).sum()}")
print(f"  Claims needing 2 searches: {(agent_results['searches_used'] == 2).sum()}")

Running agent evaluation...


Evaluating [agent]: 100%|██████████| 80/80 [03:24<00:00,  2.55s/it]

Done.

  COMPARISON: Baseline vs Agent

  Baseline
    Overall:    58.8%
    Consistent: 64.7%
    Contradict: 48.3%

  Agent
    Overall:    58.8%
    Consistent: 74.5%
    Contradict: 31.0%

  Avg searches per claim: 1.65
  Claims needing 1 search: 28
  Claims needing 2 searches: 52


In [ ]:
# ─────────────────────────────────────────────
# Multi-hop trace for deliverable
# ─────────────────────────────────────────────

multi_hop = agent_results[agent_results["searches_used"] == 2]
print(f"Claims that triggered a second search: {len(multi_hop)}")

if len(multi_hop) > 0:
    example = multi_hop.iloc[0]
    print(f"\nRunning verbose trace on:\n{example['claim'][:100]}...\n")
    verify_claim_agent(
        claim=example["claim"],
        book_name=example["book_name"],
        char=train_df.loc[example.name, "char"],
        max_searches=2,
        verbose=True,
    )

[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Claims that triggered a second search: 79
AGENT STARTING
CLAIM: Thalcave’s people faded as colonists advanced; his father, last of the tribal gu...
BOOK:  In Search of the Castaways

[Search 1] Query: 'Thalcave’s people faded as colonists advanced; his father, last of the'
  Chunk 1: "Chiefs of tribes that were very powerful thirty years ago, before they
were driven beyond the sierr...
  Chunk 2: Fortunately Thalcave solved the difficulty. This guide, who was
accustomed to conduct travelers alon...
  Chunk 3: "And to what does Thalcave attribute this abandonment?"

"He cannot tell; he is astonished. That is ...


[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Confidence check: 'INSUFFICIENT
The answer is correct because the claim includes specific details not mentioned in the passages.' → INSUFFICIENT ✗


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  Confidence check: 'INSUFFICIENT

The passages do not provide enough specific information to verify or contradict the claim. While' → INSUFFICIENT ✗


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Refined query: '__________
The user is asking for a search query that focuses on the most specific name, date, or event in the claim. The claim is'

[Search 3] Query: '__________
The user is asking for a search query that focuses on the m'
  Chunk 1: dates that agreed, and striking particulars. But details, however exact
they may be, do not constitu...
  Chunk 2: "First of all," continued Glenarvan, "we must consider three distinct
points in this document. First...
  Chunk 3: "Exactly," replied the major.

[Sidenote: "LINE UPON LINE."]

"What can we conjecture?" resumed Glen...


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Confidence check: 'INSUFFICIENT

The provided passages do not contain enough specific information to verify or contradict the claim.' → INSUFFICIENT ✗

  [Max searches reached — proceeding with best available context]

[Final Verdict] CONSISTENT ✅
Raw: 1
Answer: 0
The answer is 0.
The reason is:
The claim states that Thalcave's people faded as colonis

── FULL LOG TRACE ──

Step 1 — SEARCH
  Query:     Thalcave’s people faded as colonists advanced; his father, last of the tribal guides, knew the pampas geography and animal ways, while his mother died giving birth. Boyhood was spent roaming the plains with his father, learning to track, tame horses and steer by the stars.
  Retrieved: 3 chunks
  Sufficient: False

Step 1 — REFINE_QUERY
  New query: "Thalcave's caziques Calfoucoura, Catriel, and Yanchetruz"
Answer: "Thalcave's

Step 2 — SEARCH
  Query:     "Thalcave's caziques Calfoucoura, Catriel, and Yanchetruz"
Answer: "Thalcave's
  Retrieved: 3 chunks
  Sufficient: False

Step 2 — 